# ZRP_Build with Optional Geocoding Enabled
The purpose of this notebook is to explore ZRP_Build's memory and time utilization. 

The purpose of this notebook is to demonstrates how to use the `ZRP_Build`, with **optional geocoding** support. With this feature, users can provide geo identifiers (such as census tract or block group) in the input dataset, and ZRP will automatically leverage them for buiding race proxying models or proxying race and ethnicity.  

By specifying the geo identifier names when initializing `ZRP_Build`, users can streamline the process without needing to re-geocode records with existing geo identifiers. This ensures a more efficient and flexible workflow, especially when working with datasets that already include census geo information.

In [1]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi=False

In [2]:
from os.path import join, expanduser
import pandas as pd
import copy
import sys
import os
import re
import warnings
from time import time

## Set source code path here

In [3]:
warnings.filterwarnings(action='once')
home = expanduser('~')

src_path = os.getcwd()
root = os.path.join(src_path, "../..")
sys.path.append(src_path)

In [4]:
from zrp import ZRP
from zrp.modeling import ZRP_Build
from zrp.prepare.utils import load_file

## Load data 
When leveraging optional geocoding we recommend providing the following columns to ensure all records can be used for model development: first_name, middle_name, last_name, house_number, street_address, city, state, zip_code, census_tract, block_group, and race.

The unique identifier can be provided in the index or in the column space, by default here it is provided in the index.

In [5]:
zrp_sample = pd.read_parquet(DATA_FILE_PATH)
zrp_sample = zrp_sample[['first_name', 'middle_name', 'last_name', 'house_number',
                         'street_address', 'city', 'state', 'zip_code', 
                         'GEOID_ZIP', 'GEOID_CT', 'GEOID_BG', 'race']]
zrp_sample = zrp_sample.drop_duplicates()

### **Defining geo identifiers: census tract and block group**  

Geo identifiers such as **census tract** and **block group** uniquely define geographic areas. These geo identifiers provide a more granular view than zip code, which has been found to improve accuracy when proxying race and ethnicity. These identifiers are composed of FIPS (Federal Information Processing Standards) codes, which provide a standardized way to refer to locations.

#### Census tract format  
A census tract is a subdivision of a county used for statistical purposes. It is defined by combining the following FIPS codes:  
1. **State FIPS code** (2 digits) – Identifies the state.  
2. **County FIPS code** (3 digits) – Identifies the county within the state.  
3. **Census tract code** (6 digits) – Identifies the tract within the county.  

The full **census tract ID** is a *string of 11 characters* formatted as:  `SSCCCCTTTTTT`
- **Example census tract ID:** `"06001400100"`  
  - `"06"` = California (state FIPS)  
  - `"001"` = Alameda County (county FIPS)  
  - `"400100"` = census tract 4001.00**  

---

#### Block group format  
A block group is a further subdivision within a census tract. It is defined by adding a block group code (1 digit) to the census tract ID.  

The full block group ID is a *string of 12 characters* formatted as:  `SSCCCCTTTTTTB`
- **Example block group ID:** `"060014001002"`  
  - `"06"` = California (state FIPS)  
  - `"001"` = Alameda County (county FIPS)  
  - `"400100"` = census tract 4001.00  
  - `"2"` = block group 2  

---

**Note**
Leading zeros are critical and must be retained by storing the values as *strings*

### Data Preview

In [6]:
zrp_sample.dtypes.value_counts()

object    12
dtype: int64

In [7]:
zrp_sample.describe().loc['top', :]

first_name               JAMES
middle_name                  L
last_name              JOHNSON
house_number               105
street_address       N MAIN ST
city                  COLUMBIA
state                       SC
zip_code                 29072
GEOID_ZIP                29483
GEOID_CT           45019004608
GEOID_BG          450570112021
race                     WHITE
Name: top, dtype: object

## Invoke the `ZRP_Build` on the sample data
Initialize `ZRP_Build` with optional geocoding by specifying census tract and block group identifiers from the dataset.  

Additionally to make it easier to track experiments, we recommend defining a `file_path` to control where intermediate artifacts (pipeline, model, and supporting data) are stored and using a model name to uniquely identify the model build.  

In [8]:
zest_race_predictor = ZRP_Build(file_path='artifacts_optional_geocode', zrp_model_name='zrpbuild',census_tract='GEOID_CT', block_group='GEOID_BG', model_dev=True)
zest_race_predictor.fit()

### Build Models
Call `.transform` to start processing the data and building ZRP models. Monitor progress with built-in printouts that track each step, including the XGBoost model building progress messages. During execution, all progress messages will be displayed in the output of the transform cell, providing real-time feedback, similar to how XGBoost reports validation progress during model training.  


In [9]:
%%time
zrp_output = zest_race_predictor.transform(zrp_sample.sample(1000))

  0%|          | 0/166 [00:00<?, ?it/s]

####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 1000
     Is key unique: True
   [Completed] Validating input data

   Formatting P1
   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['SC']
   ... on state: SC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=-1)]: Done  38 tasks      | elapsed:    0.0s
100%|██████████| 166/166 [00:00<00:00, 7947.75it/s]
[Parallel(n_jobs=-1)]: Done 166 out of 166 | elapsed:    0.0s finished


      ...replicating address
         ...Base
         ...Number processing...
         House number dataframe expansion is complete! (n=166)
         ...Base
         ...Map street suffixes...
         ...Mapped & split by street suffixes...
         ...Number processing...

         Address dataframe expansion is complete! (n=211)
      ...formatting
   [Completed] Processing geo data
   [Start] Mapping geo data
      ...merge user input & lookup table
      ...mapping


100%|██████████| 1/1 [00:03<00:00,  3.38s/it]

...

[Pipeline] .. (step 5 of 8) Processing Name Aggregation, total=   0.1s
[Pipeline] . (step 6 of 8) Processing Drop Features (2), total=   0.0s
[Pipeline] ............ (step 7 of 8) Processing Impute, total=   0.0s


100%|██████████| 1/1 [00:00<00:00, 1429.06it/s]

[Pipeline]  (step 8 of 8) Processing Correlated Feature Selection, total=   0.3s
Directory already exists

---
Transforming FE data



[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished



---
Saving FE data
...Output saved

---
building zrp_model

 training data shape:800,172

---
fitting zrp_model... n_class=5
[0]	train-merror:0.13942	train-WeightedAUC:-0.75171
Multiple eval metrics have been passed: 'train-WeightedAUC' will be used for early stopping.

Will train until train-WeightedAUC hasn't improved in 2000 rounds.
[1]	train-merror:0.13237	train-WeightedAUC:-0.79811
[2]	train-merror:0.12780	train-WeightedAUC:-0.83899
[3]	train-merror:0.12480	train-WeightedAUC:-0.85861
[4]	train-merror:0.11672	train-WeightedAUC:-0.87609
[5]	train-merror:0.11202	train-WeightedAUC:-0.88876
[6]	train-merror:0.10967	train-WeightedAUC:-0.91056
[7]	train-merror:0.10432	train-WeightedAUC:-0.92132
[8]	train-merror:0.10171	train-WeightedAUC:-0.93497
[9]	train-merror:0.09689	train-WeightedAUC:-0.94317
[10]	train-merror:0.08933	train-WeightedAUC:-0.95475
[11]	train-merror:0.08672	train-WeightedAUC:-0.95883
[12]	train-merror:0.08176	train-WeightedAUC:-0.96357
[13]	train-merror:0.07655	train-We

*Note*: The following output data frame is not evaluated for performance/accuracy. As stated above, this notebook trains on an insignificant amount of training data for the purpose of demonstrating quickly how to use ZRP Build. A larger dataset is necessary to build a model with strong performance.